In [23]:
import torch
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset
from sklearn.model_selection import train_test_split

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device:{device}")

Device:cuda


In [9]:
torch.manual_seed(100)

In [11]:
df = pd.read_csv("fashion-mnist_train.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
df.shape



(60000, 785)

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, label to pixel784
dtypes: int64(785)
memory usage: 359.3 MB


In [15]:
df.columns

Index(['label', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5', 'pixel6',
       'pixel7', 'pixel8', 'pixel9',
       ...
       'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779', 'pixel780',
       'pixel781', 'pixel782', 'pixel783', 'pixel784'],
      dtype='str', length=785)

In [17]:
df["label"].unique()

array([2, 9, 6, 0, 3, 4, 5, 8, 7, 1])

In [19]:
x = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [24]:
x_train, x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [26]:
print(x_train.shape)
print(x_test.shape)

(48000, 784)
(12000, 784)


In [27]:
x_train = x_train/255.0
x_test = x_test/255.0

In [28]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features,dtype=torch.float32).float()
        self.labels = torch.tensor(labels,dtype=torch.long)
        
    def __len__(self):
        return self.features.shape[0]
    def __getitem__(self,index):
        return self.features[index],self.labels[index]
    
       
    

In [29]:
train_dataset = CustomDataset(x_train,y_train)
test_dataset = CustomDataset(x_test,y_test)

In [31]:
train_dataset[1]

(tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0

In [30]:
train_loader = DataLoader(train_dataset,batch_size= 32 ,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True,pin_memory=True)

In [42]:
len(train_loader)

1500

In [36]:
class FashionClassifierNet(nn.Module):
    
    def __init__(self, num_features):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(64,10)
        )
        
    def forward(self,x):
        return self.model(x)
    

In [39]:
epochs = 100
learning_rate = 0.1
loss_func = nn.CrossEntropyLoss()
model = FashionClassifierNet(x.shape[1])
model = model.to(device)
optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

In [43]:
for epoch in range(epochs):
    each_epoch_total_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        
        output = model(batch_features)
        batch_loss = loss_func(output,batch_labels)
        
        optimizer.zero_grad()
        
        batch_loss.backward()
        
        optimizer.step()
        
        each_epoch_total_loss = each_epoch_total_loss + batch_loss.item()
        
    avg_loss = each_epoch_total_loss/len(train_loader)
    print(f"epoch: {epoch +1} and loss : {avg_loss}")
        

epoch: 1 and loss : 0.29600239263723294
epoch: 2 and loss : 0.29814456153909363
epoch: 3 and loss : 0.29840637613336246
epoch: 4 and loss : 0.290695809426407
epoch: 5 and loss : 0.2936625756174326
epoch: 6 and loss : 0.2923295800884565
epoch: 7 and loss : 0.2901753462528189
epoch: 8 and loss : 0.2895118615354101
epoch: 9 and loss : 0.28941963099315765
epoch: 10 and loss : 0.28737180975824594
epoch: 11 and loss : 0.29209148875872293
epoch: 12 and loss : 0.2866174167469144
epoch: 13 and loss : 0.28523249462246897
epoch: 14 and loss : 0.28190941335012515
epoch: 15 and loss : 0.2820628420164188
epoch: 16 and loss : 0.2837738305230935
epoch: 17 and loss : 0.28145877426614363
epoch: 18 and loss : 0.2813374089722832
epoch: 19 and loss : 0.2792387661039829
epoch: 20 and loss : 0.27982236735274396
epoch: 21 and loss : 0.27994734108199676
epoch: 22 and loss : 0.2788333418220282
epoch: 23 and loss : 0.2746494474808375
epoch: 24 and loss : 0.2768598530292511
epoch: 25 and loss : 0.2774956734081109

In [50]:
# evaluation on test data
model.eval()

total = 0
current = 0
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        
        output = model(batch_features)
        _,predict = torch.max(output,1)
        
        
        total = total + batch_labels.size(0)
        current = current + (predict == batch_labels).sum().item()
    print(f'Accuracy: {(current/total)}')

Accuracy: 0.8935


In [49]:
# evaluation on training data
model.eval()

total = 0
current = 0
with torch.no_grad():
    for batch_features, batch_labels in train_loader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        
        output = model(batch_features)
        _,predict = torch.max(output,1)
        
        
        total = total + batch_labels.size(0)
        current = current + (predict == batch_labels).sum().item()
    print(f'Accuracy: {(current/total)}')

Accuracy: 0.9485416666666666
